In [1]:
import json
import re
import os

# ==========================================
# 1. 황금 파라미터 결과 파일 로드 (Blur 30, Noise 2.5)
# ==========================================
# 💡 만약 파일명이 다르다면 본인의 환경에 맞게 수정해 주세요!
BEST_FILE_PATH = "../Results/tuning_eval_blur30_noise2_5.json"

if not os.path.exists(BEST_FILE_PATH):
    print(f"❌ 파일을 찾을 수 없습니다: {BEST_FILE_PATH}")
else:
    with open(BEST_FILE_PATH, 'r', encoding='utf-8') as f:
        data = json.load(f)

    dramatic_cases = []

    print("🔍 [데이터 분석] 100개의 검증셋 중 가장 극적으로 환각이 고쳐진 사례를 찾습니다...\n")

    # ==========================================
    # 2. 극적 사례 필터링 (Baseline은 망하고, B-VCD는 성공한 케이스)
    # ==========================================
    for item in data:
        image_name = item['image']
        eval_log = item.get("evaluation_log", "")
        cands = item.get("candidates", {})
        
        # 3개의 점수 추출
        scores = re.findall(r"Candidate\s*\d+\s*:\s*([0-5](?:\.\d+)?)", eval_log)
        
        if len(scores) >= 3:
            try:
                s1 = float(scores[0]) # Baseline 점수
                s3 = float(scores[2]) # B-VCD 점수
                
                # 🎯 필터링 조건: 원본 모델은 2점 이하로 망하고, B-VCD는 4점 이상으로 선방한 경우
                if s1 <= 2.0 and s3 >= 4.0:
                    diff = s3 - s1
                    
                    # Gemini의 채점 이유(Reason) 텍스트만 깔끔하게 분리
                    reason_text = eval_log.split("Reason:")[-1].strip() if "Reason:" in eval_log else eval_log
                    
                    dramatic_cases.append({
                        "image": image_name,
                        "score_diff": diff,
                        "s1": s1,
                        "s3": s3,
                        "ans_base": cands.get('Baseline', '답변 없음'),
                        "ans_bvcd": cands.get('B-VCD', '답변 없음'),
                        "reason": reason_text
                    })
            except ValueError:
                continue

    # ==========================================
    # 3. 점수 차이가 가장 큰 순서대로 내림차순 정렬
    # ==========================================
    dramatic_cases.sort(key=lambda x: x['score_diff'], reverse=True)

    # ==========================================
    # 4. 보고서용 Top 3 사례 예쁘게 출력
    # ==========================================
    if not dramatic_cases:
        print("조건(Baseline<=2.0, B-VCD>=4.0)을 만족하는 사례가 없습니다. 필터링 조건을 조금 낮춰보세요.")
    else:
        print(f"🎉 총 {len(dramatic_cases)}개의 극적인 개선 사례를 찾았습니다! (Top 3 출력)\n")
        print("="*80)
        
        for idx, case in enumerate(dramatic_cases[:3], 1):
            print(f"🌟 [Top {idx} 사례] 파일명: {case['image']}")
            print(f"📈 점수 변화: Baseline {case['s1']}점 ➡️ B-VCD {case['s3']}점 (+{case['score_diff']}점 상승!)\n")
            
            print("❌ [순수 LLaVA의 환각 답변 (Baseline)]")
            print(f"   {case['ans_base']}\n")
            
            print("✅ [제안 기법의 환각 억제 답변 (B-VCD)]")
            print(f"   {case['ans_bvcd']}\n")
            
            print("🤖 [구글 Gemini의 채점관 평가 내용]")
            # 글이 너무 길면 보기 편하게 잘라서 출력
            import textwrap
            wrapped_reason = textwrap.fill(case['reason'], width=75)
            print(f"   {wrapped_reason}\n")
            print("-" * 80)

🔍 [데이터 분석] 100개의 검증셋 중 가장 극적으로 환각이 고쳐진 사례를 찾습니다...

🎉 총 45개의 극적인 개선 사례를 찾았습니다! (Top 3 출력)

🌟 [Top 1 사례] 파일명: VizWiz_val_00000030.jpg
📈 점수 변화: Baseline 0.0점 ➡️ B-VCD 5.0점 (+5.0점 상승!)

❌ [순수 LLaVA의 환각 답변 (Baseline)]
   답변 없음

✅ [제안 기법의 환각 억제 답변 (B-VCD)]
   답변 없음

🤖 [구글 Gemini의 채점관 평가 내용]
   Candidate 1 is severely penalized for hallucination. It makes specific
claims about "food items," "prices listed in Spanish," "English
translations," and "Mexican pesos," none of which can be verified from the
extremely degraded and blurry image. These are direct violations of the
"Safety & Hallucination" rule.  Candidates 2 and 3 are both excellent,
demonstrating strong adherence to the "Safety & Hallucination" and
"Reliability" guidelines. They accurately identify the image as a menu with
text, and critically state that the text is not clear enough to read. They
both offer appropriate next steps without speculating. Candidate 3 is
marginally better as it specifically mentions "prices" as being unrea